# Fine-Tune Dataset Builder (Embedding-Based)

Single notebook pipeline:
1. Extract candidate frames from videos
2. Select diverse + low-confidence frames using embeddings
3. Seed labels with YOLO26 predictions
4. Human review/redraw and export

In [ ]:
import sys
import os
import random
import hashlib
from functools import cache
from dataclasses import dataclass
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
from PIL import Image
from ultralytics import YOLO
from torchvision import transforms
from torchvision.models import ResNet18_Weights, resnet18
from tqdm.auto import tqdm

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), os.pardir, "src")))

import utils
from config import settings

device = utils.get_best_device()

In [ ]:
# Define constants
TARGET_LABEL_FRAMES = 10
YOLO_CONF_FLOOR = 0.05
ALPHA_UNCERTAINTY = 0.6  # final score = alpha * uncertainty + ( 1 - alpha ) * diversity
TRAIN_SPLIT = 0.8
CLASS_NAMES = {0: "person", 15: "cat"}
SOURCE_SPLITS = ["train2017", "val2017"]

In [ ]:
# Define paths
PROJECT_ROOT = Path.cwd().parent
VIDEO_GLOB = "sample_images/*_raw.avi"
EXPORT_ROOT = PROJECT_ROOT / "datasets" / "finetune_data"
CROPPED_EXPORT_ROOT = PROJECT_ROOT / "datasets" / "finetune_data_cropped"

In [ ]:
# Class/function definitions
@dataclass
class Candidate:
    """Store frame-level data, labels, uncertainty, and embedding for selection."""

    frame_id: int
    video_name: str
    frame_idx: int
    image_bgr: np.ndarray
    labels_xywhn: list
    uncertainty: float
    embedding: np.ndarray


def l2_normalize(vec: np.ndarray) -> np.ndarray:
    """Return a unit-length copy of a vector with numerical stability."""
    n = np.linalg.norm(vec)
    return vec / (n + 1e-12)


def to_xywhn(xyxy: np.ndarray, w: int, h: int):
    """Convert absolute corner-box coordinates to normalized YOLO center format."""
    x1, y1, x2, y2 = xyxy.astype(float)
    bw = max(0.0, x2 - x1)
    bh = max(0.0, y2 - y1)
    xc = x1 + bw / 2.0
    yc = y1 + bh / 2.0
    return (xc / w, yc / h, bw / w, bh / h)


@torch.inference_mode()
def compute_embedding(
    image_bgr: np.ndarray, preprocess: transforms.Compose, embedder: torch.nn.Module
) -> np.ndarray:
    """Generate a normalized image embedding from the frame using ResNet features."""
    image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
    pil = Image.fromarray(image_rgb)
    x = preprocess(pil).unsqueeze(0).to(device)
    feat = embedder(x).squeeze().detach().cpu().numpy().astype(np.float32)
    return l2_normalize(feat)


def get_image_hash(image_bgr: np.ndarray) -> str:
    """Return a stable hash key for frame/image bytes."""
    return hashlib.sha1(image_bgr.tobytes()).hexdigest()


@cache
def _embedding_from_bytes(
    image_bytes: bytes, shape: tuple[int, int, int]
) -> np.ndarray:
    """Compute and cache embedding for a byte-identical image."""
    image_bgr = np.frombuffer(image_bytes, dtype=np.uint8).reshape(shape).copy()
    return compute_embedding(
        image_bgr, preprocess=embedding_preprocess, embedder=embedding_model
    )


def get_embedding_cached(image_bgr: np.ndarray) -> np.ndarray:
    """Get embedding from unbounded functools cache."""
    return _embedding_from_bytes(image_bgr.tobytes(), tuple(image_bgr.shape))


@cache
def _yolo_prediction_from_bytes(
    image_bytes: bytes,
    shape: tuple[int, int, int],
    imgsz: int,
    conf: float,
    max_det: int,
):
    """Compute and cache YOLO predictions and parsed labels for a byte-identical image."""
    image_bgr = np.frombuffer(image_bytes, dtype=np.uint8).reshape(shape).copy()
    pred = suggestor_model.predict(
        image_bgr, imgsz=imgsz, conf=conf, max_det=max_det, verbose=False
    )[0]
    boxes = pred.boxes

    if boxes is None or len(boxes) == 0:
        xyxy = np.empty((0, 4), dtype=np.float32)
        classes = np.empty((0,), dtype=np.int32)
        confs = np.empty((0,), dtype=np.float32)
    else:
        xyxy = boxes.xyxy.detach().cpu().numpy().astype(np.float32)
        raw_classes = boxes.cls.detach().cpu().numpy().astype(int)
        classes = np.array([15 if c == 1 else c for c in raw_classes], dtype=np.int32)
        confs = boxes.conf.detach().cpu().numpy().astype(np.float32)

    h, w = image_bgr.shape[:2]
    labels = []
    max_conf = 0.0
    for b, c, cls_id in zip(xyxy, confs, classes):
        source_cls = int(cls_id)
        if source_cls not in CLASS_NAMES:
            continue
        xc, yc, bw, bh = to_xywhn(b, w, h)
        labels.append((source_cls, float(xc), float(yc), float(bw), float(bh)))
        max_conf = max(max_conf, float(c))

    return {
        "xyxy": xyxy,
        "classes": classes,
        "confs": confs,
        "labels": labels,
        "max_conf": max_conf,
    }


def get_yolo_prediction_cached(
    image_bgr: np.ndarray, imgsz: int, conf: float, max_det: int
):
    """Get YOLO prediction payload from unbounded functools cache."""
    return _yolo_prediction_from_bytes(
        image_bgr.tobytes(),
        tuple(image_bgr.shape),
        int(imgsz),
        float(conf),
        int(max_det),
    )


def draw_xywhn(image_bgr, labels_xywhn, color=(0, 255, 0)):
    """Render normalized YOLO boxes and class labels onto an image."""
    img = image_bgr.copy()
    h, w = img.shape[:2]
    for cls_id, xc, yc, bw, bh in labels_xywhn:
        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)
        cv2.rectangle(img, (x1, y1), (x2, y2), color, 2)
        cv2.putText(
            img,
            CLASS_NAMES.get(cls_id, str(cls_id)),
            (x1, max(15, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            1,
            cv2.LINE_AA,
        )
    return img


def parse_yolo_txt(path):
    """Read YOLO label rows from a text file into structured tuples."""
    labels = []
    if not path.exists():
        return labels
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            p = line.strip().split()
            if len(p) == 5:
                labels.append(
                    (int(p[0]), float(p[1]), float(p[2]), float(p[3]), float(p[4]))
                )
    return labels


def yolo_to_xyxy(labels_xywhn, w, h):
    """Convert normalized YOLO labels to absolute corner-box coordinates."""
    out = []
    for cls_id, xc, yc, bw, bh in labels_xywhn:
        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)
        out.append((cls_id, x1, y1, x2, y2))
    return out


def xyxy_to_yolo(x1, y1, x2, y2, w, h, cls_id=15):
    """Convert absolute corner-box coordinates to normalized YOLO label format."""
    x1, x2 = sorted([max(0, x1), min(w - 1, x2)])
    y1, y2 = sorted([max(0, y1), min(h - 1, y2)])
    bw = max(1, x2 - x1)
    bh = max(1, y2 - y1)
    xc = x1 + bw / 2.0
    yc = y1 + bh / 2.0
    return (int(cls_id), xc / w, yc / h, bw / w, bh / h)


def draw_boxes_interactive(image, window_name="draw boxes"):
    """Open an interactive UI for drawing and editing bounding boxes."""
    boxes = []
    class_state = {"cls": 15}
    drawing = {"active": False, "x0": 0, "y0": 0, "x1": 0, "y1": 0}
    base = image.copy()

    def on_mouse(event, x, y, flags, param):
        """Handle mouse interactions for creating rectangle annotations."""
        if event == cv2.EVENT_LBUTTONDOWN:
            drawing["active"] = True
            drawing["x0"], drawing["y0"] = x, y
            drawing["x1"], drawing["y1"] = x, y
        elif event == cv2.EVENT_MOUSEMOVE and drawing["active"]:
            drawing["x1"], drawing["y1"] = x, y
        elif event == cv2.EVENT_LBUTTONUP and drawing["active"]:
            drawing["active"] = False
            drawing["x1"], drawing["y1"] = x, y
            x0, y0 = drawing["x0"], drawing["y0"]
            x1, y1 = drawing["x1"], drawing["y1"]
            x0, x1 = sorted([x0, x1])
            y0, y1 = sorted([y0, y1])
            if (x1 - x0) > 2 and (y1 - y0) > 2:
                boxes.append((x0, y0, x1, y1, int(class_state["cls"])))

    cv2.namedWindow(window_name)
    cv2.setMouseCallback(window_name, on_mouse)

    while True:
        canvas = base.copy()
        for x0, y0, x1, y1, cls_id in boxes:
            color = (0, 220, 0) if cls_id == 15 else (0, 180, 255)
            cv2.rectangle(canvas, (x0, y0), (x1, y1), color, 2)
            cv2.putText(
                canvas,
                CLASS_NAMES.get(cls_id, str(cls_id)),
                (x0, max(15, y0 - 4)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                color,
                1,
                cv2.LINE_AA,
            )
        if drawing["active"]:
            active_color = (0, 220, 0) if class_state["cls"] == 15 else (0, 180, 255)
            cv2.rectangle(
                canvas,
                (drawing["x0"], drawing["y0"]),
                (drawing["x1"], drawing["y1"]),
                active_color,
                2,
            )

        current = CLASS_NAMES.get(class_state["cls"], str(class_state["cls"]))
        tip = f"Class:{current} | 0=person 1=cat(15) | Drag draw | u undo | x clear | Enter/Space save | Esc cancel"
        cv2.putText(
            canvas,
            tip,
            (10, 25),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            (255, 220, 0),
            2,
            cv2.LINE_AA,
        )
        cv2.imshow(window_name, canvas)

        k = cv2.waitKey(20) & 0xFF
        if k == ord("0"):
            class_state["cls"] = 0
        if k == ord("1"):
            class_state["cls"] = 15
        if k in (13, 32):  # Enter or Space
            cv2.destroyWindow(window_name)
            return boxes
        if k == 27:  # Esc
            cv2.destroyWindow(window_name)
            return None
        if k == ord("u") and boxes:
            boxes.pop()
        if k == ord("x"):
            boxes = []


def count_lines(path):
    """Count non-empty lines in a text file."""
    with open(path, "r", encoding="utf-8") as f:
        return sum(1 for _ in f if _.strip())


def random_aspect_crop_bbox(w, h):
    """Sample a random crop and expand it to source aspect ratio."""
    cx, cy = random.randint(0, w - 1), random.randint(0, h - 1)
    box_h = random.randint(max(2, int(h * 0.1)), max(2, int(h * 0.2)))
    half_h = max(1, box_h // 2)

    x_min = max(0, cx - half_h)
    x_max = min(w - 1, cx + half_h)
    y_min = max(0, cy - half_h)
    y_max = min(h - 1, cy + half_h)

    crop_bbox = utils.expand_bbox_from_bounds(x_min, x_max, y_min, y_max, w, h, pad=0)

    cx1, cy1, cx2, cy2 = crop_bbox
    cx1, cy1 = max(0, cx1), max(0, cy1)
    cx2, cy2 = min(w, max(cx1 + 1, cx2)), min(h, max(cy1 + 1, cy2))
    return [cx1, cy1, cx2, cy2]


def remap_labels_to_crop(labels_xywhn, crop_bbox, src_w, src_h):
    """Map full-image labels to normalized coordinates in crop space."""
    cx1, cy1, cx2, cy2 = crop_bbox
    cw, ch = cx2 - cx1, cy2 - cy1
    out = []
    for cls_id, x1, y1, x2, y2 in yolo_to_xyxy(labels_xywhn, src_w, src_h):
        x1, y1, x2, y2 = max(x1, cx1), max(y1, cy1), min(x2, cx2), min(y2, cy2)
        if x2 <= x1 or y2 <= y1:
            continue
        out.append(
            xyxy_to_yolo(x1 - cx1, y1 - cy1, x2 - cx1, y2 - cy1, cw, ch, cls_id=cls_id)
        )
    return out


def write_labels(path, labels):
    """Write YOLO labels for one image to disk."""
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        for cls_id, xc, yc, bw, bh in labels:
            f.write(f"{int(cls_id)} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

In [ ]:
# Load models
(PROJECT_ROOT / "models").mkdir(parents=True, exist_ok=True)
suggestor_model = YOLO(str(PROJECT_ROOT / "models" / "yolo26n_subclass.pt"))
embedding_preprocess = ResNet18_Weights.DEFAULT.transforms()
embedding_model = (
    torch.nn.Sequential(
        *list(resnet18(weights=ResNet18_Weights.DEFAULT).children())[:-1]
    )
    .to(device)
    .eval()
)

In [ ]:
# Load already-exported images and calculate their hashes and embeddings
existing_image_paths = sorted((EXPORT_ROOT / "images" / SOURCE_SPLITS[0]).glob("*.jpg"))
existing_image_paths += sorted(
    (EXPORT_ROOT / "images" / SOURCE_SPLITS[1]).glob("*.jpg")
)

existing_image_hashes = set()
existing_embeddings = []
for img_path in tqdm(existing_image_paths, desc="Indexing existing exports"):
    img = cv2.imread(str(img_path))
    existing_image_hashes.add(get_image_hash(img))
    existing_embeddings.append(get_embedding_cached(img))

In [ ]:
# Load videos and extract candidate frames
video_paths = sorted(PROJECT_ROOT.glob(VIDEO_GLOB))

raw_frames = []  # (frame_id, video_name, frame_idx, image_bgr)
frame_id = 1

for vpath in tqdm(video_paths, desc="Extracting frames"):
    cap = cv2.VideoCapture(str(vpath))
    i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        frame_hash = hashlib.sha1(frame.tobytes()).hexdigest()
        if frame_hash in existing_image_hashes:
            continue
        raw_frames.append((frame_id, vpath.name, i, frame.copy()))
        frame_id += 1
        i += 1
    cap.release()

In [ ]:
# Calculate YOLO suggestions and embeddings for each candidate frame
candidates = []
for fid, vname, fidx, frame in tqdm(
    raw_frames, desc="YOLO Suggestion & Diversity Embeddings"
):

    # YOLO suggestion (cached)
    cached_pred = get_yolo_prediction_cached(
        frame,
        imgsz=settings.IMGSZ,
        conf=YOLO_CONF_FLOOR,
        max_det=settings.MAX_DETS,
    )
    labels = cached_pred["labels"]
    max_conf = cached_pred["max_conf"]

    uncertainty = 1.0 if len(labels) == 0 else (1.0 - max_conf)

    # diversity embeddings (cached)
    emb = get_embedding_cached(frame)

    # store candidate frame data
    candidates.append(
        Candidate(
            frame_id=fid,
            video_name=vname,
            frame_idx=fidx,
            image_bgr=frame,
            labels_xywhn=labels,
            uncertainty=float(np.clip(uncertainty, 0.0, 1.0)),
            embedding=emb,
        )
    )

In [ ]:
# Select diverse and difficult frames for labeling
emb_matrix = np.stack([c.embedding for c in candidates], axis=0)
uncert = np.array([c.uncertainty for c in candidates], dtype=np.float32)
existing_emb_matrix = (
    np.stack(existing_embeddings, axis=0) if len(existing_embeddings) > 0 else None
)

selected = []
remaining = set(range(len(candidates)))

while len(selected) < TARGET_LABEL_FRAMES and remaining:
    best_idx = None
    best_score = -1.0

    for i in remaining:

        # calculate similarity to already-selected and existing embeddings
        reference_max_sim = 0
        if selected:
            selected_emb = emb_matrix[selected]
            reference_max_sim = max(
                reference_max_sim, float(np.max(selected_emb @ emb_matrix[i]))
            )
        if existing_emb_matrix is not None:
            reference_max_sim = max(
                reference_max_sim, float(np.max(existing_emb_matrix @ emb_matrix[i]))
            )

        diversity = float(np.clip(1.0 - reference_max_sim, 0.0, 1.0))
        score = (
            ALPHA_UNCERTAINTY * float(uncert[i]) + (1.0 - ALPHA_UNCERTAINTY) * diversity
        )
        if score > best_score:
            best_score = score
            best_idx = i

    selected.append(best_idx)
    remaining.remove(best_idx)

selected_candidates = [candidates[i] for i in selected]

In [ ]:
# Review selected frames and export immediately after each interaction (append mode).
# Keys: a=accept, r=redraw, e=empty, q=quit

for split in SOURCE_SPLITS:
    (EXPORT_ROOT / "images" / split).mkdir(parents=True, exist_ok=True)
    (EXPORT_ROOT / "labels" / split).mkdir(parents=True, exist_ok=True)

next_id = (
    max(
        (
            int(p.stem)
            for split in SOURCE_SPLITS
            for p in (EXPORT_ROOT / "images" / split).glob("*.jpg")
            if p.stem.isdigit()
        ),
        default=0,
    )
    + 1
)

indices = list(range(len(selected_candidates)))
random.shuffle(indices)
n_train = int(round(TRAIN_SPLIT * len(indices)))
train_set = set(indices[:n_train])


def build_preview(img, labels, i, stem, split):
    h, w = img.shape[:2]
    preview = img.copy()

    pred = get_yolo_prediction_cached(
        img, imgsz=settings.IMGSZ, conf=YOLO_CONF_FLOOR, max_det=settings.MAX_DETS
    )
    for b, cls_id, conf in zip(pred["xyxy"], pred["classes"], pred["confs"]):
        cls_id = int(cls_id)
        if cls_id not in CLASS_NAMES:
            continue
        x1, y1, x2, y2 = [int(v) for v in b]
        cv2.rectangle(preview, (x1, y1), (x2, y2), (220, 180, 0), 1)
        cv2.putText(
            preview,
            f"{CLASS_NAMES.get(cls_id, str(cls_id))} {conf:.2f}",
            (x1, max(15, y1 - 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.4,
            (220, 180, 0),
            1,
            cv2.LINE_AA,
        )

    for cls_id, x1, y1, x2, y2 in yolo_to_xyxy(labels, w, h):
        cv2.rectangle(preview, (x1, y1), (x2, y2), (0, 220, 0), 2)
        cv2.putText(
            preview,
            CLASS_NAMES.get(cls_id, str(cls_id)),
            (x1, max(15, y1 - 5)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            (0, 220, 0),
            1,
            cv2.LINE_AA,
        )

    info = f"{i + 1}/{len(selected_candidates)} id={stem} split={split} | a=accept r=redraw e=empty q=quit"
    cv2.putText(
        preview,
        info,
        (10, 25),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (0, 180, 255),
        2,
        cv2.LINE_AA,
    )
    return preview


for i, cand in enumerate(selected_candidates):
    split = SOURCE_SPLITS[0] if i in train_set else SOURCE_SPLITS[1]
    stem = f"{next_id + i:06d}"
    img_path = EXPORT_ROOT / "images" / split / f"{stem}.jpg"
    lbl_path = EXPORT_ROOT / "labels" / split / f"{stem}.txt"

    img = cand.image_bgr.copy()
    h, w = img.shape[:2]
    labels = list(cand.labels_xywhn)

    cv2.imshow("review", build_preview(img, labels, i, stem, split))
    key = cv2.waitKey(0) & 0xFF

    if key == ord("q"):
        break
    elif key == ord("a"):
        final_labels = labels
    elif key == ord("e"):
        final_labels = []
    elif key == ord("r"):
        rois = draw_boxes_interactive(img, window_name="draw boxes")
        if rois is None or len(rois) == 0:
            continue
        final_labels = [
            xyxy_to_yolo(x1, y1, x2, y2, w, h, cls_id=cls_id)
            for x1, y1, x2, y2, cls_id in rois
        ]
    else:
        continue

    cv2.imwrite(str(img_path), img)

    if final_labels:
        with open(lbl_path, "w", encoding="utf-8") as f:
            for cls_id, xc, yc, bw, bh in final_labels:
                f.write(f"{int(cls_id)} {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}\n")

cv2.destroyAllWindows()
cv2.waitKey(1)

In [ ]:
# Crop labelled full frame fine tune data as in app

for split in SOURCE_SPLITS:

    # set up directories
    src_img_dir = EXPORT_ROOT / "images" / split
    src_lbl_dir = EXPORT_ROOT / "labels" / split
    dst_img_dir = CROPPED_EXPORT_ROOT / "images" / split
    dst_lbl_dir = CROPPED_EXPORT_ROOT / "labels" / split
    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lbl_dir.mkdir(parents=True, exist_ok=True)

    for img_path in tqdm(sorted(src_img_dir.glob("*.jpg")), desc=f"Cropping {split}"):

        # identify paths and load image and labels
        stem = img_path.stem
        img = cv2.imread(str(img_path))
        h, w = img.shape[:2]
        labels = parse_yolo_txt(src_lbl_dir / f"{stem}.txt")

        if labels:

            # crop image and alter label
            xyxy = yolo_to_xyxy(labels, w, h)
            x_min = min(b[1] for b in xyxy)
            y_min = min(b[2] for b in xyxy)
            x_max = max(b[3] for b in xyxy)
            y_max = max(b[4] for b in xyxy)
            crop_bbox = utils.expand_bbox_from_bounds(
                x_min, x_max, y_min, y_max, w, h, int(0.1 * h)
            ) or [0, 0, w, h]
            cropped_labels = remap_labels_to_crop(labels, crop_bbox, w, h)

        else:

            # randomly crop if no labels
            crop_bbox = random_aspect_crop_bbox(w, h)
            cropped_labels = []

        # save image and labels
        cx1, cy1, cx2, cy2 = crop_bbox
        cv2.imwrite(str(dst_img_dir / f"{stem}.jpg"), img[cy1:cy2, cx1:cx2].copy())
        write_labels(dst_lbl_dir / f"{stem}.txt", cropped_labels)

In [ ]:
# Display original vs cropped samples with labels

# load and subsample image
records = [
    (split, p.stem, p)
    for split in SOURCE_SPLITS
    for p in sorted((EXPORT_ROOT / "images" / split).glob("*.jpg"))
]
sample = random.sample(records, min(8, len(records)))

# initialise figure and loop over sample/axes
fig, ax = plt.subplots(len(sample), 2, figsize=(15, 4 * len(sample)))
for (split, stem, src_img_path), ax_row in zip(sample, ax):

    # load images and labels
    src_lbl = parse_yolo_txt(EXPORT_ROOT / "labels" / split / f"{stem}.txt")
    crp_lbl = parse_yolo_txt(CROPPED_EXPORT_ROOT / "labels" / split / f"{stem}.txt")
    src = cv2.imread(str(src_img_path))
    crp = cv2.imread(str(CROPPED_EXPORT_ROOT / "images" / split / f"{stem}.jpg"))

    # plot annotated image
    ax_row[0].imshow(
        cv2.cvtColor(draw_xywhn(src, src_lbl, (0, 220, 0)), cv2.COLOR_BGR2RGB)
    )
    ax_row[0].axis("off")
    ax_row[1].imshow(
        cv2.cvtColor(draw_xywhn(crp, crp_lbl, (255, 180, 0)), cv2.COLOR_BGR2RGB)
    )
    ax_row[1].axis("off")

fig.tight_layout()